Parcial final Electiva 1 | Realizado por: Jonhatan Stiven Gutiérrez Vélez

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA electiva;

/* Creación del contenedor de almacenamiento para los archivos planos */
CREATE VOLUME IF NOT EXISTS workspace.electiva.parcial_3;

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA electiva;

/* Aseguramos que el volumen exista */
CREATE VOLUME IF NOT EXISTS workspace.electiva.parcial_3;

/* 1. Categorías y Productos */
CREATE OR REPLACE TABLE workspace.electiva.product_categories AS
SELECT * 
FROM read_files(
    '/Volumes/workspace/electiva/parcial_3/productcategories.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories AS
SELECT * 
FROM read_files(
    '/Volumes/workspace/electiva/parcial_3/productsubcategories.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.products AS
SELECT * 
FROM read_files(
    '/Volumes/workspace/electiva/parcial_3/products.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

/* 2. Clientes, Empleados y Proveedores */
CREATE OR REPLACE TABLE workspace.electiva.customers AS
SELECT * 
FROM read_files(
    '/Volumes/workspace/electiva/parcial_3/customers.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.employee AS
SELECT * 
FROM read_files(
    '/Volumes/workspace/electiva/parcial_3/employees.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.vendor AS
SELECT * 
FROM read_files(
    '/Volumes/workspace/electiva/parcial_3/vendors.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

/* 3. Órdenes y Relación Proveedor-Producto */
CREATE OR REPLACE TABLE workspace.electiva.vendor_product AS
SELECT * 
FROM read_files(
    '/Volumes/workspace/electiva/parcial_3/vendorproduct.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.orders AS
SELECT * 
FROM read_files(
    '/Volumes/workspace/electiva/parcial_3/orders.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

 Punto 1

In [0]:
%sql
-- Punto 1: user_name de cada cliente (id-XX-yy-AW)
select
    customerid,
    concat(firstname, ' ', lastname) as fullname,
    concat(
        cast(customerid as string), '-',
        upper(substring(firstname, 1, 2)), '-',
        lower(substring(lastname, 1, 2)), '-AW'
    ) as user_name
from workspace.electiva.customers;

Punto 2

In [0]:
%sql
-- Tabla donde guardo el user_name de cada cliente
create table if not exists workspace.electiva.page_user (
    customerid int,
    user_name string
)
using delta
comment 'customerid y user_name generado para cada cliente';

In [0]:
%sql
-- Lleno page_user con la misma fórmula del punto 1
insert into workspace.electiva.page_user
select
    cast(customerid as int) as customerid,
    concat(
        cast(customerid as string), '-',
        upper(substring(firstname, 1, 2)), '-',
        lower(substring(lastname, 1, 2)), '-AW'
    ) as user_name
from workspace.electiva.customers;

In [0]:
%sql
-- Reviso que sí se hayan cargado los registros
select * from workspace.electiva.page_user;

Punto 3

In [0]:
%sql
-- Punto 3: vista con las órdenes de agosto de 2011 y el descuento según la categoría
create or replace view workspace.electiva.`v_utl_ordenes_mitad_año` as
with ordenes_agosto as (
    select
        p.productname,
        p.modelname,
        pu.user_name,
        o.salesorderid,
        pc.categoryname,
        cast(o.orderdate as date) as orderdate,
        cast(o.linetotal as double) as linetotal,
        case
            when pc.categoryname ilike 'Bikes'       then 0.10
            when pc.categoryname ilike 'Clothing'    then 0.15
            when pc.categoryname ilike 'Components'  then 0.08
            when pc.categoryname ilike 'Accessories' then 0.20
            else 0
        end as `descuento_mitad_año`
    from workspace.electiva.orders as o
    inner join workspace.electiva.page_user as pu
        on o.customerid = pu.customerid
    inner join workspace.electiva.products as p
        on o.productid = p.productid
    -- left join para no perder productos sin subcategoría o categoría
    left join workspace.electiva.product_sub_categories as psc
        on p.subcategoryid = psc.subcategoryid
    left join workspace.electiva.product_categories as pc
        on psc.categoryid = pc.categoryid
    where year(cast(o.orderdate as date)) = 2011
      and month(cast(o.orderdate as date)) = 8
)
select
    productname,
    modelname,
    user_name,
    salesorderid,
    categoryname,
    orderdate,
    linetotal,
    `descuento_mitad_año`,
    linetotal - (linetotal * `descuento_mitad_año`) as total_descuentos_aplicados
from ordenes_agosto;

-- Miro cómo quedó la vista
select * from workspace.electiva.`v_utl_ordenes_mitad_año`;

Punto 4

In [0]:
-- Punto 4: vista materializada customer_class
-- Clasifico a los clientes según cuántas órdenes hicieron en 2011:
-- 3 o más = Oro, 2 = Plata, 1 = Ocasional
create or replace materialized view workspace.electiva.customer_class as
select
    count(distinct o.salesorderid) as ammount_of_orders,
    pu.user_name,
    concat(c.firstname, ' ', c.lastname) as customerfullname,
    case
        when count(distinct o.salesorderid) >= 3 then 'Oro'
        when count(distinct o.salesorderid) = 2  then 'Plata'
        else 'Ocasional'
    end as user_level,
    c.customerid
from workspace.electiva.orders as o
inner join workspace.electiva.customers as c
    on o.customerid = c.customerid
inner join workspace.electiva.page_user as pu
    on c.customerid = pu.customerid
where year(cast(o.orderdate as date)) = 2011
group by
    pu.user_name,
    c.firstname,
    c.lastname,
    c.customerid;

-- Reviso el resultado
select * from workspace.electiva.customer_class;